In [1]:
!nvidia-smi

Thu Apr  9 13:15:10 2026       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.261.03             Driver Version: 535.261.03   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA H100 80GB HBM3          On  | 00000000:19:00.0 Off |                    0 |
| N/A   42C    P0             121W / 700W |  30453MiB / 81559MiB |      0%      Default |
|                                         |                      |             Disabled |
+-----------------------------------------+----------------------+--

In [3]:
from llama_cpp import Llama

llm = Llama.from_pretrained(
    repo_id="Jackrong/Qwen3.5-27B-Claude-4.6-Opus-Reasoning-Distilled-GGUF",
    filename="*Q4_K_M.gguf",
    n_ctx=24576,
    n_gpu_layers=-1,  # offload all layers to GPU
    verbose=False,
)
print("model loaded")

llama_context: n_ctx_seq (24576) < n_ctx_train (262144) -- the full capacity of the model will not be utilized


model loaded


In [4]:
#testing the big daddy model

In [5]:
output = llm.create_chat_completion(
    messages=[{"role": "user", "content": "Extract microbe-disease relationships: Patients with Crohn's disease showed reduced Faecalibacterium prausnitzii (p<0.01). Return JSON only."}],
    temperature=0,
    max_tokens=512,
)
print(output["choices"][0]["message"]["content"])

Let me analyze the given text to extract microbe-disease relationships:

Text: "Patients with Crohn's disease showed reduced Faecalibacterium prausnitzii (p<0.01)."

Step 1: Identify the disease
- "Crohn's disease" is clearly mentioned as the disease condition

Step 2: Identify the microbe
- "Faecalibacterium prausnitzii" is a bacterium (microbe)

Step 3: Determine the relationship
- The text states that patients with Crohn's disease "showed reduced" levels of this microbe
- This indicates a negative association - the microbe is decreased in the disease state
- The p-value (p<0.01) indicates statistical significance

Step 4: Format as JSON
- I need to create a JSON structure that captures:
  - The disease name
  - The microbe name
  - The type of relationship (reduced/decreased)
  - The statistical significance

The relationship can be described as:
- Disease: Crohn's disease
- Microbe: Faecalibacterium prausnitzii
- Relationship: reduced (negative association)
- Significance: p<0.01
<

In [8]:
#running the same prompt from the llama version 

In [ ]:
#original prompt
'''
PROMPT_TEMPLATE = """Extract all microbe-disease relationships from this paper.

For each microbe with a reported change, record:
- taxon_name: exact name from the paper
- direction: "increased" or "decreased"

Return ONLY valid JSON:
{{"disease": "disease name", "taxa_enriched": ["taxon1", "taxon2"], "taxa_depleted": ["taxon3", "taxon4"]}}

If no clear microbe change reported, return:
{{"disease": "disease name", "taxa_enriched": [], "taxa_depleted": []}}

Paper text:
{text}

JSON:"""
'''


In [ ]:
#rigid prompt

"""
Extract all microbe-disease relationships from this paper.
For each microbe with a reported change, record:
- taxon_name: exact name from the paper
- direction: "increased" or "decreased"

Rules:
- Only read Abstract, Results, and Discussion sections. Ignore Methods, figures, and tables.
- Do NOT include microbes only mentioned in citations or background literature.
- Do NOT infer directionality if the paper does not explicitly state it.
- Do NOT include microbes that were experimentally administered (e.g. probiotics given as treatment).
- Be forgiving on taxon name spelling but strict on directionality — if direction is ambiguous, omit the taxon.

Return ONLY valid JSON, no commentary:
{{"disease": "disease name", "taxa_enriched": ["taxon1", "taxon2"], "taxa_depleted": ["taxon3", "taxon4"]}}
If no clear microbe changes reported, return:
{{"disease": "disease name", "taxa_enriched": [], "taxa_depleted": []}}
Paper text:
{text}
JSON:
"""


In [11]:
import json, time, os, csv, re
from datetime import datetime

DATA_PATH = "gold_standard_final_15.json"
os.makedirs("benchmark_results", exist_ok=True)

with open(DATA_PATH, "r") as f:
    papers = json.load(f)

PROMPT_TEMPLATE = '''

'''

def smart_truncate(text):
    for marker in ["References\n", "REFERENCES\n", "Bibliography\n"]:
        idx = text.rfind(marker)
        if idx > 0:
            return text[:idx]
    return text

results = []

for i, paper in enumerate(papers):
    print(f"[{i+1}/{len(papers)}] {paper['title'][:70]}...")

    text = smart_truncate(paper["text"])
    prompt = PROMPT_TEMPLATE.format(text=text)
    start = time.time()

    output = llm.create_chat_completion(
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=2048,
    )
    raw = output["choices"][0]["message"]["content"]
    elapsed = time.time() - start

    try:
        clean = raw
        # strip <think> block if present
        if "</think>" in clean:
            clean = clean.split("</think>")[-1]
        clean = clean.strip()
        if "```json" in clean:
            clean = clean.split("```json")[1].split("```")[0]
        elif "```" in clean:
            clean = clean.split("```")[1].split("```")[0]
        parsed = json.loads(clean.strip())
    except:
        parsed = {"disease": paper["disease"], "taxa_enriched": [], "taxa_depleted": [], "parse_error": True}

    results.append({
        "title": paper["title"],
        "disease": paper["disease"],
        "in_gold_standard": paper["in_gold_standard"],
        "expected_enriched": paper.get("taxa_enriched", ""),
        "expected_depleted": paper.get("taxa_depleted", ""),
        "predicted_enriched": ", ".join(parsed.get("taxa_enriched", [])),
        "predicted_depleted": ", ".join(parsed.get("taxa_depleted", [])),
        "predicted_disease": parsed.get("disease", ""),
        "time_seconds": round(elapsed, 2),
        "parse_error": parsed.get("parse_error", False),
    })

    print(f"  enriched: {parsed.get('taxa_enriched', [])}")
    print(f"  depleted: {parsed.get('taxa_depleted', [])}")
    print(f"  {elapsed:.1f}s")

print(f"\nDone. {len(results)} papers.")

[1/15] Intestinal flora induces depression by mediating the dysregulation of ...
  enriched: []
  depleted: []
  21.5s
[2/15] Gut microbes exacerbate systemic inflammation and behavior disorders i...
  enriched: []
  depleted: []
  32.4s
[3/15] Alterations in gut microbiota and metabolomic profiles in acute stroke...
  enriched: ['Firmicutes', 'Faecalibacterium', 'Agathobacter', 'Blautia', 'Barnesiella', 'Dialister', 'Ruminococcus']
  depleted: ['Bacteroidota', 'Bacteroides', 'Escherichia-Shigella']
  31.9s
[4/15] Gut microbiome dysbiosis across early Parkinson's disease, REM sleep b...
  enriched: ['Collinsella', 'Desulfovibrio', 'Oscillospiraceae_UCG-005', 'Oscillospiraceae_UCG-002', 'Akkermansia', 'UBA1819', 'Family_XIII_AD3011_group', 'Uncultured_Oscillospiraceae_g061']
  depleted: ['Butyricicoccus', 'Faecalibacterium', 'Roseburia', 'Lachnospiraceae_ND3007_group', 'Lachnospira', '[Eubacterium]_ventriosum_group', 'Lachnospiraceae', 'Lachnoclostridium', '[Ruminococcus]_torques_group'

In [ ]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

json_path = f"benchmark_results/qwopus_updatedPrompt_6April_{timestamp}.json"
with open(json_path, "w") as f:
    json.dump(results, f, indent=2)

csv_path = f"benchmark_results/qwopus_{timestamp}.csv"
cols = ["title", "disease", "in_gold_standard",
        "expected_enriched", "expected_depleted",
        "predicted_enriched", "predicted_depleted",
        "predicted_disease", "time_seconds", "parse_error"]

with open(csv_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=cols)
    writer.writeheader()
    writer.writerows(results)

print(f"JSON: {json_path}")
print(f"CSV:  {csv_path}")

In [4]:
#if want to run eval on custom file
import csv
import re
with open("benchmark_results/qwopus_cot_updatedprompt_20260329_050629.csv", "r") as f:
    results = list(csv.DictReader(f))

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def parse_taxa(val):
    if not val or str(val) == "NaN" or str(val) == "nan":
        return []
    taxa = re.split(r'[,;]', str(val))
    cleaned = []
    for t in taxa:
        t = re.sub(r'\(.*?\)', '', t).strip().lower()
        t = re.sub(r'p\s*[<>=]\s*[\d.]+', '', t).strip()
        t = t.strip('.) ')
        if t and t != "nan" and len(t) > 2:
            cleaned.append(t)
    return cleaned

def match_taxa(predicted, expected):
    if not predicted and not expected:
        return [], []
    if not predicted:
        return [], expected
    if not expected:
        return [(p, "N/A (no expected)", 0.0) for p in predicted], []

    all_names = predicted + expected
    tfidf = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4)).fit_transform(all_names)
    sim = cosine_similarity(tfidf[:len(predicted)], tfidf[len(predicted):])

    matches = []
    matched_expected_idx = set()
    for i, pred in enumerate(predicted):
        best_j = sim[i].argmax()
        score = round(float(sim[i][best_j]), 3)
        matches.append((pred, expected[best_j], score))
        if score >= 0.5:
            matched_expected_idx.add(best_j)

    missed = [expected[j] for j in range(len(expected)) if j not in matched_expected_idx]
    return matches, missed

for r in results:
    print(f"\n{'='*60}")
    print(f"{r['title'][:70]}")
    print(f"Disease: {r['disease']} | Gold: {r['in_gold_standard']}")

    pred_enr = parse_taxa(r["predicted_enriched"])
    pred_dep = parse_taxa(r["predicted_depleted"])
    exp_enr = parse_taxa(r["expected_enriched"])
    exp_dep = parse_taxa(r["expected_depleted"])

    enr_matches, enr_missed = match_taxa(pred_enr, exp_enr)
    dep_matches, dep_missed = match_taxa(pred_dep, exp_dep)

    if enr_matches:
        print("  ENRICHED:")
        for pred, exp, score in enr_matches:
            flag = "✅" if score >= 0.5 else "❌"
            print(f"    {flag} {pred} → {exp}  sim={score}")
    if enr_missed:
        print("  MISSED ENRICHED:")
        for m in enr_missed:
            print(f"    ⚠️  {m}")

    if dep_matches:
        print("  DEPLETED:")
        for pred, exp, score in dep_matches:
            flag = "✅" if score >= 0.5 else "❌"
            print(f"    {flag} {pred} → {exp}  sim={score}")
    if dep_missed:
        print("  MISSED DEPLETED:")
        for m in dep_missed:
            print(f"    ⚠️  {m}")

    if not enr_matches and not dep_matches and not enr_missed and not dep_missed:
        print("  (no predictions, no expected)")

    tp = sum(1 for _, _, s in enr_matches + dep_matches if s >= 0.5)
    fp = sum(1 for _, _, s in enr_matches + dep_matches if s < 0.5)
    fn = len(enr_missed) + len(dep_missed)
    print(f"  → TP={tp}  FP={fp}  FN={fn}")


Intestinal flora induces depression by mediating the dysregulation of 
Disease: Stroke | Gold: No
  MISSED ENRICHED:
    ⚠️  g_erysipelotrichaceae ucg-003
  → TP=0  FP=0  FN=1

Gut microbes exacerbate systemic inflammation and behavior disorders i
Disease: Other | Gold: Yes
  ENRICHED:
    ✅ porphyromonas → porphyromonas  sim=1.0
    ✅ akkermansia → akkermansia  sim=1.0
    ✅ eubacterium siraeum → eubacterium siraeum  sim=1.0
    ✅ akkermansia muciniphila → akkermansia muciniphila  sim=1.0
    ✅ fusobacterium varium → fusobacterium varium  sim=1.0
    ✅ megasphaera elsdenii → megasphaera elsdenii  sim=1.0
    ✅ clostridium aldenense → clostridium aldenense  sim=1.0
    ✅ bacteroides sp. om05-12 → bacteroides sp. om05-12  sim=1.0
    ✅ streptococcus sp. a12 → streptococcus sp. a12  sim=1.0
  DEPLETED:
    ✅ sutterella → sutterella  sim=1.0
    ✅ roseburia faecis → roseburia faecis  sim=1.0
    ✅ eubacterium eligens → eubacterium eligens  sim=1.0
    ✅ sutterella parvirubra → sutterella

In [6]:
import json, time, os, csv, re
from datetime import datetime

DATA_PATH = "gold_standard_final_15.json"
os.makedirs("benchmark_results", exist_ok=True)

with open(DATA_PATH, "r") as f:
    papers = json.load(f)

PROMPT_TEMPLATE = """Extract all microbe-disease relationships from this paper.

For each microbe with a reported change, record:
- taxon_name: exact name from the paper
- direction: "increased" or "decreased"

Before answering, carefully scan the Abstract, Results, and Discussion sections. Extract at EVERY taxonomic level mentioned (phylum, family, genus, species). Do not miss microbes that appear only in Results tables or only in Discussion. If the same microbe appears at multiple taxonomic levels, include all of them.

Return ONLY valid JSON:
{{"disease": "disease name", "taxa_enriched": ["taxon1", "taxon2"], "taxa_depleted": ["taxon3", "taxon4"]}}

If no clear microbe changes vs healthy control, return:
{{"disease": "disease name", "taxa_enriched": [], "taxa_depleted": []}}

Paper text:
{text}

JSON:"""

def smart_truncate(text):
    for marker in ["References\n", "REFERENCES\n", "Bibliography\n"]:
        idx = text.rfind(marker)
        if idx > 0:
            return text[:idx]
    return text

results = []

for i, paper in enumerate(papers):
    print(f"[{i+1}/{len(papers)}] {paper['title'][:70]}...")

    text = smart_truncate(paper["text"])
    prompt = PROMPT_TEMPLATE.format(text=text)
    start = time.time()

    output = llm.create_chat_completion(
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=4096,
    )
    raw = output["choices"][0]["message"]["content"]
    elapsed = time.time() - start

    try:
        clean = raw
        if "</think>" in clean:
            clean = clean.split("</think>")[-1]
        clean = clean.strip()
        if "```json" in clean:
            clean = clean.split("```json")[1].split("```")[0]
        elif "```" in clean:
            clean = clean.split("```")[1].split("```")[0]
        parsed = json.loads(clean.strip())
    except:
        parsed = {"disease": paper["disease"], "taxa_enriched": [], "taxa_depleted": [], "parse_error": True}

    results.append({
        "title": paper["title"],
        "disease": paper["disease"],
        "in_gold_standard": paper["in_gold_standard"],
        "expected_enriched": paper.get("taxa_enriched", ""),
        "expected_depleted": paper.get("taxa_depleted", ""),
        "predicted_enriched": ", ".join(parsed.get("taxa_enriched", [])),
        "predicted_depleted": ", ".join(parsed.get("taxa_depleted", [])),
        "predicted_disease": parsed.get("disease", ""),
        "time_seconds": round(elapsed, 2),
        "parse_error": parsed.get("parse_error", False),
    })

    print(f"  enriched: {parsed.get('taxa_enriched', [])}")
    print(f"  depleted: {parsed.get('taxa_depleted', [])}")
    print(f"  {elapsed:.1f}s")

print(f"\nDone. {len(results)} papers.")

[1/15] Intestinal flora induces depression by mediating the dysregulation of ...


NameError: name 'llm' is not defined

In [14]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

json_path = f"benchmark_results/qwopus_cot_updatedprompt_{timestamp}.json"
with open(json_path, "w") as f:
    json.dump(results, f, indent=2)

csv_path = f"benchmark_results/qwopus_cot_updatedprompt_{timestamp}.csv"
cols = ["title", "disease", "in_gold_standard",
        "expected_enriched", "expected_depleted",
        "predicted_enriched", "predicted_depleted",
        "predicted_disease", "time_seconds", "parse_error"]

with open(csv_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=cols)
    writer.writeheader()
    writer.writerows(results)

print(f"JSON: {json_path}")
print(f"CSV:  {csv_path}")

JSON: benchmark_results/qwopus_cot_updatedprompt_20260329_050629.json
CSV:  benchmark_results/qwopus_cot_updatedprompt_20260329_050629.csv


In [15]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def parse_taxa(val):
    if not val or str(val) == "NaN" or str(val) == "nan":
        return []
    taxa = re.split(r'[,;]', str(val))
    cleaned = []
    for t in taxa:
        t = re.sub(r'\(.*?\)', '', t).strip().lower()
        t = re.sub(r'p\s*[<>=]\s*[\d.]+', '', t).strip()
        t = t.strip('.) ')
        if t and t != "nan" and len(t) > 2:
            cleaned.append(t)
    return cleaned

def match_taxa(predicted, expected):
    if not predicted and not expected:
        return [], []
    if not predicted:
        return [], expected
    if not expected:
        return [(p, "N/A (no expected)", 0.0) for p in predicted], []

    all_names = predicted + expected
    tfidf = TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4)).fit_transform(all_names)
    sim = cosine_similarity(tfidf[:len(predicted)], tfidf[len(predicted):])

    matches = []
    matched_expected_idx = set()
    for i, pred in enumerate(predicted):
        best_j = sim[i].argmax()
        score = round(float(sim[i][best_j]), 3)
        matches.append((pred, expected[best_j], score))
        if score >= 0.5:
            matched_expected_idx.add(best_j)

    missed = [expected[j] for j in range(len(expected)) if j not in matched_expected_idx]
    return matches, missed

total_tp = total_fp = total_fn = 0

for r in results:
    print(f"\n{'='*60}")
    print(f"{r['title'][:70]}")
    print(f"Disease: {r['disease']} | Gold: {r['in_gold_standard']}")

    pred_enr = parse_taxa(r["predicted_enriched"])
    pred_dep = parse_taxa(r["predicted_depleted"])
    exp_enr = parse_taxa(r["expected_enriched"])
    exp_dep = parse_taxa(r["expected_depleted"])

    enr_matches, enr_missed = match_taxa(pred_enr, exp_enr)
    dep_matches, dep_missed = match_taxa(pred_dep, exp_dep)

    if enr_matches:
        print("  ENRICHED:")
        for pred, exp, score in enr_matches:
            flag = "✅" if score >= 0.5 else "❌"
            print(f"    {flag} {pred} → {exp}  sim={score}")
    if enr_missed:
        print("  MISSED ENRICHED:")
        for m in enr_missed:
            print(f"    ⚠️  {m}")

    if dep_matches:
        print("  DEPLETED:")
        for pred, exp, score in dep_matches:
            flag = "✅" if score >= 0.5 else "❌"
            print(f"    {flag} {pred} → {exp}  sim={score}")
    if dep_missed:
        print("  MISSED DEPLETED:")
        for m in dep_missed:
            print(f"    ⚠️  {m}")

    if not enr_matches and not dep_matches and not enr_missed and not dep_missed:
        print("  (no predictions, no expected)")

    tp = sum(1 for _, _, s in enr_matches + dep_matches if s >= 0.5)
    fp = sum(1 for _, _, s in enr_matches + dep_matches if s < 0.5)
    fn = len(enr_missed) + len(dep_missed)
    total_tp += tp
    total_fp += fp
    total_fn += fn
    print(f"  → TP={tp}  FP={fp}  FN={fn}")

p = total_tp / (total_tp + total_fp) if (total_tp + total_fp) else 0
r = total_tp / (total_tp + total_fn) if (total_tp + total_fn) else 0
f1 = 2 * p * r / (p + r) if (p + r) else 0

print(f"\n{'='*60}")
print(f"OVERALL — Qwopus CoT")
print(f"TP={total_tp}  FP={total_fp}  FN={total_fn}")
print(f"Precision: {p:.3f}")
print(f"Recall:    {r:.3f}")
print(f"F1:        {f1:.3f}")


Intestinal flora induces depression by mediating the dysregulation of 
Disease: Stroke | Gold: No
  MISSED ENRICHED:
    ⚠️  g_erysipelotrichaceae ucg-003
  → TP=0  FP=0  FN=1

Gut microbes exacerbate systemic inflammation and behavior disorders i
Disease: Other | Gold: Yes
  ENRICHED:
    ✅ porphyromonas → porphyromonas  sim=1.0
    ✅ akkermansia → akkermansia  sim=1.0
    ✅ eubacterium siraeum → eubacterium siraeum  sim=1.0
    ✅ akkermansia muciniphila → akkermansia muciniphila  sim=1.0
    ✅ fusobacterium varium → fusobacterium varium  sim=1.0
    ✅ megasphaera elsdenii → megasphaera elsdenii  sim=1.0
    ✅ clostridium aldenense → clostridium aldenense  sim=1.0
    ✅ bacteroides sp. om05-12 → bacteroides sp. om05-12  sim=1.0
    ✅ streptococcus sp. a12 → streptococcus sp. a12  sim=1.0
  DEPLETED:
    ✅ sutterella → sutterella  sim=1.0
    ✅ roseburia faecis → roseburia faecis  sim=1.0
    ✅ eubacterium eligens → eubacterium eligens  sim=1.0
    ✅ sutterella parvirubra → sutterella